# Part 1 — NLP: turning annual-report text into numbers

**What this notebook does.** It takes the raw text of 763 annual reports and turns it into three
numerical representations that a model can learn from. It does no modelling itself.

**In → out**

| In | Out |
| --- | --- |
| `report_sections_1/2.jsonl.gz` — the MD&A, directors' report, auditor's report and CARO annexure of each report | `nlp_output/embeddings_mdna.npy` — one 768-number vector per sentence |
| `documents_labeled.csv` — which company-year each report belongs to, and whether it ended in insolvency | `nlp_output/embeddings_auditor.npy` — the same for the auditor's report |
| `ratios.csv`, `language_features.csv` | `nlp_output/modelling_table.csv` — one row per report: label, fold, ratios, language features, perplexity |

Everything in `nlp_output/` is what **Part 2 (deep learning)** reads. The two notebooks share
nothing else, so you can re-run Part 2 as often as you like without touching this one.

**Final version (3 October 2026).** Two inputs changed after the progress report and both must
be in the data folder: `language_features.csv` with the **corrected CARO flags** (from
`final_run/bpp_final/data_v2/`; the old rule read line-wrapped denials such as "has not / defaulted"
as defaults) and **`folds.csv`**, the frozen fold of every report (Section 4). Outputs are cleared:
re-run on Colab.

**Runtime:** about 5 minutes if the embeddings are already cached in `_cache/`, or about 25 minutes
the first time. Needs a **T4 GPU** (Runtime → Change runtime type).

## 1 · Setup

Mount Google Drive and point at the dataset folder. Nothing here is specific to this project
except the path.

In [ ]:
!pip -q install "transformers>=4.40" 2>&1 | tail -1
!git clone -q https://github.com/ananya-baweja/bankruptcy-prediction.git /content/bpp-repo 2>/dev/null || true

import sys, gzip, json, re, time, numpy as np, pandas as pd, torch
sys.path.insert(0, "/content/bpp-repo/src")      # for bpp.nlp.lm (standard library only)
from pathlib import Path
from google.colab import drive
from sklearn.model_selection import GroupKFold

drive.mount("/content/drive")
DATA = Path("/content/drive/MyDrive/DL data")    # <<< edit if your folder is elsewhere
OUT  = DATA/"nlp_output";  OUT.mkdir(exist_ok=True)
CACHE= DATA/"_cache";      CACHE.mkdir(exist_ok=True)
dev  = "cuda" if torch.cuda.is_available() else "cpu"
print("GPU:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU, then re-run."

## 2 · Load the report text

Each line of the two `.jsonl.gz` files is one annual report, already split into its four sections
by the extraction pipeline (Phase 2). We use two of them:

- **`mdna`** — Management Discussion and Analysis. What the company says about its own year.
- **`auditor_report`** — the independent auditor's report, including any going-concern warning.

The other two sections (directors' report, CARO annexure) are in the file but are not used by the
model; the CARO flags already reached us as engineered features.

In [ ]:
rows = []
for part in ["report_sections_1.jsonl.gz", "report_sections_2.jsonl.gz"]:
    with gzip.open(DATA/part, "rt", encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            rows.append({"doc_id": r["doc_id"],
                         "mdna": r.get("mdna") or "",
                         "auditor_report": r.get("auditor_report") or ""})
text = pd.DataFrame(rows)
print(f"{len(text)} reports in the text export")
print(f"  with an MD&A section:            {int((text.mdna.str.len()>0).sum())}")
print(f"  with an auditor's report:        {int((text.auditor_report.str.len()>0).sum())}")
print(f"  median MD&A length:              {int(text.mdna.str.len().median()):,} characters")

## 3 · Build the modelling table

One row per annual report. Three things are joined on:

1. **The label** — did this company enter insolvency? Plus the *horizon*: `t-1` means this is the
   last report published before the insolvency reference date, `t-2` the one before, and so on.
2. **The accounting ratios** — eleven of them, from the financial statements.
3. **The engineered language features** — described in Section 5.

**Which rows we keep, and why.** This is the part that protects the result from being nonsense.
A report is excluded if it was published within 180 days of the insolvency admission (by then the
outcome is public), if a human reviewer found it discussing a petition against the company itself,
or if the exchange filed it under the wrong year. We *do* keep healthy companies' reports whose
matched distressed partner was excluded, because a healthy company's report cannot disclose an
insolvency that never happened.

In [ ]:
dl = pd.read_csv(DATA/"documents_labeled.csv")
lf = pd.read_csv(DATA/"language_features.csv")
ra = pd.read_csv(DATA/"ratios.csv")

RATIOS = ["current_ratio","quick_ratio","cash_to_assets","ebitda_margin","roce","roa",
          "debt_to_equity","debt_to_ebitda","interest_coverage","retained_earnings_to_assets",
          "altman_z_em"]

# keep everything the pipeline marked usable, plus the healthy partners described above
keep = dl.included | (dl.exclude_reason == "pair_partner_excluded")
base = dl.loc[keep & (dl.has_document == True),
              ["doc_id","firm_id","fy","pair_id","label","included","horizon"]].copy()

df = (base
      .merge(lf.drop(columns=[c for c in ["label","pair_id","role","horizon","included",
                                          "exclude_reason"] if c in lf.columns]),
             on=["doc_id","firm_id","fy"], how="left")
      .merge(ra[["firm_id","fy"]+RATIOS+["included_financials"]], on=["firm_id","fy"], how="left")
      .merge(text, on="doc_id", how="left")
      .fillna({"mdna":"", "auditor_report":""})
      .reset_index(drop=True))

# a flag so the model knows when the financial statements were incomplete, rather than
# silently receiving an imputed median as if it were real
df["financials_missing"] = (~df["included_financials"].fillna(False)).astype(float)
# the strict subset: rows that also satisfy the original 1:1 matched design
df["strict"] = (df.included & df.included_financials.fillna(False))

print(f"rows {len(df)}   healthy {int((df.label==0).sum())}   distressed {int((df.label==1).sum())}")
print(f"matched pairs {df.pair_id.nunique()}   strict subset {int(df.strict.sum())}")
print("\nexcluded, and why:")
print(dl.exclude_reason.value_counts(dropna=False).rename({np.nan:"(kept)"}).to_string())

In [ ]:
# The corrected CARO flags: before the fix 14% of healthy company-years carried a loan-default
# flag; after it about 6%. An old language_features.csv stops the notebook here.
rate = df.loc[df.label == 0, "caro_default_flag"].mean()
print(f"healthy company-years with a CARO loan-default flag: {rate:.1%}")
assert rate < 0.10, ("This is the language_features.csv from before the CARO correction. Copy "
                     "final_run/bpp_final/data_v2/language_features.csv into the data folder.")

## 4 · The cross-validation folds

We fix the folds **here**, in the NLP notebook, and save them in the table. That way Part 2 cannot
accidentally use a different split, and every model is compared on identical data.

**Frozen in a file.** The fold of every report is read from `folds.csv`. scikit-learn's
GroupKFold breaks ties between equally sized pairs differently across versions; a current version
would move 79% of the rows to another fold, so recomputing the folds would quietly change every
number. The cell still runs GroupKFold and prints how far it agrees with the file.

**Grouped by matched pair.** A distressed company and its matched peer are the same industry and
roughly the same size. If one ended up in the training set and the other in the test set, the model
would effectively have seen half of a matched comparison in advance. Grouping by pair prevents that.

In [ ]:
N_SPLITS = 5
y, groups = df.label.to_numpy(), df.pair_id.to_numpy()
gkf = np.full(len(df), -1)
for k, (_, te) in enumerate(GroupKFold(n_splits=N_SPLITS).split(df, y, groups)):
    gkf[te] = k
fx = pd.read_csv(DATA/"folds.csv")[["doc_id","fold"]]
df = df.merge(fx, on="doc_id", how="left")
assert df.fold.notna().all() and len(df) == len(fx), "folds.csv does not match the table"
df["fold"] = df.fold.astype(int)
fold = df.fold.to_numpy(); y, groups = df.label.to_numpy(), df.pair_id.to_numpy()
print(f"this scikit-learn's GroupKFold agrees with folds.csv on {(gkf == fold).mean():.0%} of rows")

print(df.groupby("fold").agg(rows=("label","size"), distressed=("label","sum")).to_string())
assert all(len(set(groups[fold==k]) & set(groups[fold!=k])) == 0 for k in range(N_SPLITS)), \
    "a pair appears in two folds"
print("\ncheck passed: no matched pair is split across folds")

## 5 · The engineered language features

These are the "classic" NLP features — hand-designed counts and ratios computed from the text by
`src/bpp/nlp/features.py`. They are already in `language_features.csv`; what matters is knowing
what each family measures.

| Family | What it counts | Why it might signal distress |
| --- | --- | --- |
| `*_lm_*_ratio` | Loughran–McDonald dictionary: share of words that are negative, positive, uncertain, litigious, constraining, or strong/weak modal | A finance-specific word list. Distressed firms use more negative and uncertain language |
| `*_hedge_density`, `*_hedge_phrase_density` | Hedging words and phrases ("may", "could", "subject to") | Management hedges more when it is less sure of the outcome |
| `*_distress_phrase_density` | An India-specific distress phrase list built for this project | Catches local phrasing no US dictionary covers |
| `*_fog_index`, `*_complex_word_ratio`, `*_avg_sentence_length` | Readability | Deliberately harder-to-read disclosure is a known warning sign |
| `has_going_concern`, `audit_opinion_severity`, `caro_*_flag` | The auditor's formal flags | The auditor's own assessment, stated before the event |
| `drift_*` | Year-on-year change against the same company's previous report | A company changing how it writes is more informative than its absolute tone |

Features are computed separately for the MD&A (`mdna_*`) and the auditor's report (`auditor_*`).

Two columns are dropped: `cirp_specific_mentions` and `ibc_generic_mentions` count explicit
mentions of insolvency proceedings. Removing them actually *improved* every model, which is useful
evidence that the result does not rest on reports naming their own insolvency.

In [ ]:
DROP  = {"doc_id","firm_id","fy","pair_id","role","label","horizon","included","exclude_reason",
         "perplexity_source","drift_previous_doc_id","has_previous_year","strict","fold",
         "mdna","auditor_report","included_financials"}
LEAKY = {"cirp_specific_mentions","ibc_generic_mentions"}
LANG  = [c for c in lf.columns if c not in DROP and c not in LEAKY and lf[c].dtype.kind in "fib"]

# drop anything with no information on these rows (all-missing, or the same value everywhere)
dead = [c for c in LANG+RATIOS if df[c].isna().all() or df[c].nunique(dropna=True) <= 1]
LANG = [c for c in LANG if c not in dead] + ["financials_missing"]
print(f"{len(LANG)} language features, {len(RATIOS)} ratios")
print("dropped (no information):", dead)

### 5a · A worked example — recomputing one feature from scratch

So the features are not a black box, here is the Fog index recomputed by hand for a few reports
and checked against the stored column. Fog ≈ 0.4 × (average sentence length + percentage of
complex words), where a complex word has three or more syllables.

In [ ]:
VOWELS = "aeiouy"
def syllables(word):
    w = re.sub(r"[^a-z]", "", word.lower())
    if not w: return 0
    n, prev = 0, False
    for ch in w:
        v = ch in VOWELS
        if v and not prev: n += 1
        prev = v
    if w.endswith("e") and n > 1: n -= 1
    return max(n, 1)

def fog(text):
    sents = [s for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]
    words = re.findall(r"[A-Za-z]+", text)
    if not sents or not words: return np.nan
    complex_share = 100 * sum(syllables(w) >= 3 for w in words) / len(words)
    return 0.4 * (len(words)/len(sents) + complex_share)

sample = df[df.mdna.str.len() > 2000].head(5)
print(f"{'doc_id':20s} {'recomputed':>11s} {'stored':>8s}")
for _, r in sample.iterrows():
    print(f"{r.doc_id:20s} {fog(r.mdna):11.1f} {r.mdna_fog_index:8.1f}")
print("\n(small differences are expected - the pipeline uses a syllable dictionary, this is a rule of thumb)")

## 6 · Perplexity against a healthy-firm language model

A language model trained on one kind of text can score how *surprising* a new document is. Train it
on the MD&A of healthy companies only, and a high score means "this does not read like a normal
company's report".

**Why this must be done per fold.** The reference model is built from healthy reports. If it were
fitted on *all* healthy reports and then used to score the test fold, information from the test set
would have leaked into the model. So the model is refitted five times, each time on the healthy
reports of that fold's **training** data only. This is exactly why the `perplexity` column in
`language_features.csv` is empty — the feature pipeline correctly refused to compute it without
knowing the folds.

In [ ]:
from bpp.nlp.lm import fit as lm_fit, perplexity as lm_perplexity

ppl = np.full(len(df), np.nan); t0 = time.time()
for k in range(N_SPLITS):
    train_healthy = (df.fold != k) & (df.label == 0) & (df.mdna.str.len() > 0)
    model = lm_fit(df.loc[train_healthy, "mdna"].tolist(), order=3, min_count=2)
    for i in np.flatnonzero((df.fold == k).to_numpy()):
        ppl[i] = lm_perplexity(model, df.mdna.iloc[i]) or np.nan
    print(f"  fold {k}: fitted on {int(train_healthy.sum())} healthy reports, "
          f"vocabulary {len(model.vocabulary):,}")
df["perplexity_fold"] = ppl
LANG = LANG + ["perplexity_fold"]

ok = df.perplexity_fold.notna()
print(f"\ndone in {time.time()-t0:.0f}s")
print(f"median perplexity - distressed {df.loc[ok & (df.label==1),'perplexity_fold'].median():.1f}"
      f"  |  healthy {df.loc[ok & (df.label==0),'perplexity_fold'].median():.1f}")

> **Result: this feature does not work, and that is worth reporting.** The two medians are almost
> identical and on its own the feature scores ROC-AUC ≈ 0.51 — a coin flip. The likely reason is
> that Indian MD&A sections are heavily templated: every listed company writes similar paragraphs
> about industry outlook and internal controls, so a trigram model mostly learns boilerplate that
> both groups share. We keep the column (it costs nothing and the network can ignore it) and report
> the null result honestly. In the final run, removing it from the boosting model changes PR-AUC
> by +0.003: no contribution.

## 7 · FinBERT sentence embeddings

The engineered features above throw away word order — they are counts. To let a model read the
*sentences*, each sentence is passed through **FinBERT**, a BERT model pre-trained on financial
text, which returns a 768-number vector summarising its meaning.

Three choices worth understanding:

- **The encoder is frozen.** FinBERT's 110 million weights are never updated. We run it once, store
  the vectors, and train only a small network on top. This is what makes the project fit in a free
  Colab session — about 0.36 million trainable parameters instead of 110 million.
- **Mean pooling.** A sentence's vector is the average of its word-piece vectors, ignoring padding.
- **200 sentences per report, 96 word-pieces per sentence.** Reports longer than that are cut off.

The cache is keyed by `doc_id`, so re-running this notebook costs seconds rather than minutes.

In [ ]:
from transformers import AutoTokenizer, AutoModel
MODEL_NAME, MAX_SENTS, MAX_TOKENS, DIM = "ProsusAI/finbert", 200, 96, 768

_SPLIT, _WS = re.compile(r"(?<=[.!?])\s+"), re.compile(r"\s+")
def to_sentences(t, m=MAX_SENTS):
    """Split into sentences, keep those with 5+ words (drops page numbers and table debris)."""
    out = []
    for s in _SPLIT.split(t or ""):
        s = _WS.sub(" ", s).strip()
        if len(s.split()) >= 5:
            out.append(s)
            if len(out) >= m: break
    return out

def embed(section):
    tag = f"{section}_{MODEL_NAME.split('/')[-1]}_{MAX_SENTS}x{MAX_TOKENS}"
    npy, idx = CACHE/f"embstore_{tag}.npy", CACHE/f"embstore_{tag}.json"
    have, store = {}, np.zeros((0, MAX_SENTS, DIM), np.float16)
    if npy.exists() and idx.exists():
        store = np.load(npy); have = {d:i for i,d in enumerate(json.loads(idx.read_text()))}
    sents = df[section].map(to_sentences)
    need  = [d for d in df.doc_id if d not in have]
    print(f"[{section}] {len(have)} cached, {len(need)} to embed")
    if need:
        tok = AutoTokenizer.from_pretrained(MODEL_NAME)
        enc = AutoModel.from_pretrained(MODEL_NAME).to(dev).eval().half()
        sent_of = dict(zip(df.doc_id, sents))
        new = np.zeros((len(need), MAX_SENTS, DIM), np.float16); t0 = time.time()
        with torch.no_grad():
            for j, d in enumerate(need):
                ss = sent_of[d]
                for a in range(0, len(ss), 64):                       # sub-batch within a report
                    chunk = ss[a:a+64]
                    b = tok(chunk, padding=True, truncation=True,
                            max_length=MAX_TOKENS, return_tensors="pt").to(dev)
                    h  = enc(**b).last_hidden_state                   # (n, tokens, 768)
                    mk = b["attention_mask"].unsqueeze(-1).half()
                    new[j, a:a+len(chunk)] = ((h*mk).sum(1)/mk.sum(1).clamp(min=1)).cpu().numpy()
                if j % 100 == 0: print(f"   {j}/{len(need)}  {time.time()-t0:.0f}s", flush=True)
        store = np.concatenate([store, new], 0); order = list(have.keys()) + need
        np.save(npy, store); idx.write_text(json.dumps(order))
        have = {d:i for i,d in enumerate(order)}
        del enc; torch.cuda.empty_cache()
    E = store[[have[d] for d in df.doc_id]]          # reorder to match the table, row for row
    lengths = sents.map(len).to_numpy()
    return E, lengths, sents

E_mdna, L_mdna, S_mdna = embed("mdna")
E_aud,  L_aud,  S_aud  = embed("auditor_report")
print(f"\nMD&A embeddings     {E_mdna.shape}   median {int(np.median(L_mdna))} sentences/report")
print(f"auditor embeddings  {E_aud.shape}   median {int(np.median(L_aud))} sentences/report")

## 8 · Save the handoff

Four files. Part 2 reads these and nothing else.

`sent_len_*` records how many real sentences each report had, so Part 2 can build a mask and ignore
the padding rather than treating zeros as meaningful.

In [ ]:
df["sent_len_mdna"]    = L_mdna
df["sent_len_auditor"] = L_aud

TABLE_COLS = (["doc_id","firm_id","fy","pair_id","label","horizon","included","strict","fold",
               "sent_len_mdna","sent_len_auditor"] + RATIOS + LANG)
df[TABLE_COLS].to_csv(OUT/"modelling_table.csv", index=False)
np.save(OUT/"embeddings_mdna.npy",    E_mdna)
np.save(OUT/"embeddings_auditor.npy", E_aud)
(OUT/"feature_columns.json").write_text(json.dumps(
    {"ratios": RATIOS, "language": LANG, "max_sents": MAX_SENTS, "dim": DIM,
     "n_splits": N_SPLITS, "doc_id_order": df.doc_id.tolist()}, indent=2))

print("written to", OUT)
for f in sorted(OUT.iterdir()):
    print(f"  {f.name:28s} {f.stat().st_size/1e6:8.1f} MB")
print(f"\n{len(df)} rows, {len(RATIOS)} ratios, {len(LANG)} language features")
print("Row order in the CSV matches row order in both .npy files. Part 2 relies on this.")

## 9 · What Part 2 does with this

| File | Used for |
| --- | --- |
| `modelling_table.csv` | labels, folds, ratios, language features — the tabular half of the model |
| `embeddings_mdna.npy` | the text the network reads: 763 × 200 × 768 |
| `embeddings_auditor.npy` | the same for the auditor's report, used to test whether the signal is only the auditor's flags |
| `feature_columns.json` | which columns are ratios, which are language features, and the row order |

Nothing above involves training a predictive model. That is Part 2.